In [133]:
print("Project Started")

Project Started


In [134]:
from pyspark.sql import SparkSession


In [135]:
spark = SparkSession.builder \
    .appName("Ecommerce Big Data Analytics") \
    .getOrCreate()

print("Spark Session Created")

Spark Session Created


In [136]:
orders_df = spark.read.csv(
    "../data/olist_orders_dataset.csv",
    header=True,
    inferSchema=True
)

In [137]:
orders_df.show(5)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [138]:
from pyspark.sql.functions import col, count, when

orders_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in orders_df.columns
]).show()

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [139]:
print("Total Rows:", orders_df.count())

print("Duplicate Removed Rows:",
      orders_df.dropDuplicates().count())

Total Rows: 99441
Duplicate Removed Rows: 99441


In [140]:
orders_df = orders_df.dropDuplicates()

In [141]:
from pyspark.sql.functions import to_timestamp

orders_df = orders_df.withColumn(
    "order_purchase_timestamp",
    to_timestamp("order_purchase_timestamp")
)

In [142]:
orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [143]:
print("Total Orders:", orders_df.count())

Total Orders: 99441


In [144]:
orders_df.groupBy("order_status") \
    .count() \
    .show()

+------------+-----+
|order_status|count|
+------------+-----+
|     shipped| 1107|
|    canceled|  625|
|    invoiced|  314|
|   delivered|96478|
| unavailable|  609|
|  processing|  301|
|     created|    5|
|    approved|    2|
+------------+-----+



In [145]:
import os

os.makedirs("../outputs", exist_ok=True)

In [146]:
orders_df.createOrReplaceTempView("orders")

In [147]:
spark.sql("""
SELECT COUNT(*) AS total_orders
FROM orders
""").show()

+------------+
|total_orders|
+------------+
|       99441|
+------------+



In [148]:
spark.sql("""
SELECT order_status,
COUNT(*) AS total_orders
FROM orders
GROUP BY order_status
ORDER BY total_orders DESC
""").show()

+------------+------------+
|order_status|total_orders|
+------------+------------+
|   delivered|       96478|
|     shipped|        1107|
|    canceled|         625|
| unavailable|         609|
|    invoiced|         314|
|  processing|         301|
|     created|           5|
|    approved|           2|
+------------+------------+



In [149]:
spark.sql("""
SELECT
YEAR(order_purchase_timestamp) AS year,
MONTH(order_purchase_timestamp) AS month,
COUNT(*) AS total_orders
FROM orders
GROUP BY year, month
ORDER BY year, month
""").show()

+----+-----+------------+
|year|month|total_orders|
+----+-----+------------+
|2016|    9|           4|
|2016|   10|         324|
|2016|   12|           1|
|2017|    1|         800|
|2017|    2|        1780|
|2017|    3|        2682|
|2017|    4|        2404|
|2017|    5|        3700|
|2017|    6|        3245|
|2017|    7|        4026|
|2017|    8|        4331|
|2017|    9|        4285|
|2017|   10|        4631|
|2017|   11|        7544|
|2017|   12|        5673|
|2018|    1|        7269|
|2018|    2|        6728|
|2018|    3|        7211|
|2018|    4|        6939|
|2018|    5|        6873|
+----+-----+------------+
only showing top 20 rows


In [150]:
spark.sql("""
SELECT
MONTH(order_purchase_timestamp) AS month,
COUNT(*) AS total_orders
FROM orders
GROUP BY month
ORDER BY total_orders DESC
LIMIT 10
""").show()

+-----+------------+
|month|total_orders|
+-----+------------+
|    8|       10843|
|    5|       10573|
|    7|       10318|
|    3|        9893|
|    6|        9412|
|    4|        9343|
|    2|        8508|
|    1|        8069|
|   11|        7544|
|   12|        5674|
+-----+------------+



In [151]:
monthly_sales = spark.sql("""
SELECT
YEAR(order_purchase_timestamp) AS year,
MONTH(order_purchase_timestamp) AS month,
COUNT(*) AS total_orders
FROM orders
GROUP BY year, month
ORDER BY year, month
""")

In [152]:
monthly_sales_pd = monthly_sales.toPandas()

In [153]:
monthly_sales_pd.to_csv(
    "../outputs/monthly_sales.csv",
    index=False
)

In [154]:
customers_df = spark.read.csv(
    "../data/olist_customers_dataset.csv",
    header=True,
    inferSchema=True
)

In [155]:
order_items_df = spark.read.csv(
    "../data/olist_order_items_dataset.csv",
    header=True,
    inferSchema=True
)

In [156]:
products_df = spark.read.csv(
    "../data/olist_products_dataset.csv",
    header=True,
    inferSchema=True
)

In [157]:
customers_df.show(5)
order_items_df.show(5)
products_df.show(5)

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 5 rows
+--------------------+---------

In [158]:
customers_df.createOrReplaceTempView("customers")

order_items_df.createOrReplaceTempView("order_items")

products_df.createOrReplaceTempView("products")

In [159]:
spark.sql("""
SELECT
product_id,
COUNT(*) AS total_sales
FROM order_items
GROUP BY product_id
ORDER BY total_sales DESC
LIMIT 10
""").show()

+--------------------+-----------+
|          product_id|total_sales|
+--------------------+-----------+
|aca2eb7d00ea1a7b8...|        527|
|99a4788cb24856965...|        488|
|422879e10f4668299...|        484|
|389d119b48cf3043d...|        392|
|368c6c730842d7801...|        388|
|53759a2ecddad2bb8...|        373|
|d1c427060a0f73f6b...|        343|
|53b36df67ebb7c415...|        323|
|154e7e31ebfa09220...|        281|
|3dd2a17168ec895c7...|        274|
+--------------------+-----------+



In [160]:
spark.sql("""
SELECT
product_id,
ROUND(SUM(price),2) AS revenue
FROM order_items
GROUP BY product_id
ORDER BY revenue DESC
LIMIT 10
""").show()

+--------------------+--------+
|          product_id| revenue|
+--------------------+--------+
|bb50f2e236e5eea01...| 63885.0|
|6cdd53843498f9289...| 54730.2|
|d6160fb7873f18409...|48899.34|
|d1c427060a0f73f6b...|47214.51|
|99a4788cb24856965...|43025.56|
|3dd2a17168ec895c7...| 41082.6|
|25c38557cf793876c...|38907.32|
|5f504b3a1c75b73d6...| 37733.9|
|53b36df67ebb7c415...|37683.42|
|aca2eb7d00ea1a7b8...| 37608.9|
+--------------------+--------+



In [161]:
spark.sql("""
SELECT
customer_state,
COUNT(*) AS total_customers
FROM customers
GROUP BY customer_state
ORDER BY total_customers DESC
""").show()

+--------------+---------------+
|customer_state|total_customers|
+--------------+---------------+
|            SP|          41746|
|            RJ|          12852|
|            MG|          11635|
|            RS|           5466|
|            PR|           5045|
|            SC|           3637|
|            BA|           3380|
|            DF|           2140|
|            ES|           2033|
|            GO|           2020|
|            PE|           1652|
|            CE|           1336|
|            PA|            975|
|            MT|            907|
|            MA|            747|
|            MS|            715|
|            PB|            536|
|            PI|            495|
|            RN|            485|
|            AL|            413|
+--------------+---------------+
only showing top 20 rows


In [162]:
revenue_df = spark.sql("""
SELECT
order_id,
product_id,
price,
freight_value
FROM order_items
""")

In [163]:
revenue_pd = revenue_df.toPandas()

revenue_pd.to_csv(
    "../outputs/revenue_analysis.csv",
    index=False
)

In [164]:
spark.sql("""
SELECT
ROUND(SUM(price),2) AS total_revenue
FROM order_items
""").show()

+-------------+
|total_revenue|
+-------------+
| 1.35916437E7|
+-------------+



In [165]:
spark.sql("""
SELECT
ROUND(AVG(price),2) AS avg_order_value
FROM order_items
""").show()

+---------------+
|avg_order_value|
+---------------+
|         120.65|
+---------------+

